# Genetic Algorithm for Feature Selection

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.base import clone

In [ ]:
class GeneticAlgorithm:
  def __init__(self,
               model,
               population_size=30,
               num_generations=15,
               mutation_rate=0.05,
               crossover_rate=0.8,
               random_state=None):
    self.model=model
    self.population_size=population_size
    self.num_generations=num_generations
    self.mutation_rate=mutation_rate
    self.crossover_rate=crossover_rate
    self.random_state=np.random.RandomState(random_state)

  #with soft constriant for prefered number of features
  def fitness(self, chromosome, X, y, target_k=12, penalty=.01):
      if chromosome.sum() == 0:
        return 0  # penalize empty feature sets

      selected = np.where(chromosome == 1)[0]
      model = clone(self.model)

      scores = cross_val_score(model, X[:, selected], y, cv=5)
      if target_k is not None:
        size_penalty = penalty * abs(len(selected) - target_k)
        scores -= size_penalty

      return scores.mean()

  def selection(self,population,fitness_scores,k=3):
      selected = []
      for _ in range(len(population)):
        idx = self.random_state.choice(len(population), k)
        best = idx[np.argmax(fitness_scores[idx])]
        selected.append(population[best])

      return np.array(selected)

  def crossover(self, parent1, parent2):
      if self.random_state.rand() > self.crossover_rate:
        return parent1, parent2

      point=self.random_state.randint(1, len(parent1))
      child1=np.concatenate((parent1[:point], parent2[point:]))
      child2=np.concatenate((parent2[:point], parent1[point:]))
      return child1, child2

  def mutation(self, chromosome):
      for i in range(len(chromosome)):
        if self.random_state.rand() < self.mutation_rate:
          chromosome[i] = 1 - chromosome[i]
      return chromosome

  def fit(self, X, y):
      num_features=X.shape[1]

      #intialize population
      population=self.random_state.randint(2, size=(self.population_size, num_features))

      #genetic algorithm loop
      for gen in range(self.num_generations):

        #apply fitness function to members of population
        fitness_scores = np.array([self.fitness(ind, X, y) for ind in population])

        #determine best solution
        best_idx=np.argmax(fitness_scores)
        best_chromosome=population[best_idx]
        best_score=fitness_scores[best_idx]

        print(f"Generation {gen+1}/{self.num_generations} | Best Score: {best_score:.4f}")

        #perform selction
        selection_pool=self.selection(population, fitness_scores)

        #perform crossover and mutation
        new_population=[]
        for i in range(self.population_size):
          parent1, parent2= selection_pool[i], selection_pool[(i+1)%self.population_size]
          child1, child2 = self.crossover(parent1, parent2)
          new_population.append(self.mutation(child1))
          new_population.append(self.mutation(child2))

        population =np.array(new_population)

        #perform elitism
        worst_idx=np.argmin(self.fitness(X,y,feat) for feat in population)
        population[worst_idx]=best_chromosome

      #final best solution
      final_fitness = np.array([self.fitness(ind, X, y) for ind in population])
      best_idx = np.argmax(final_fitness)
      self.best_chromosome_ = population[best_idx]
      self.best_features_ = np.where(self.best_chromosome_ == 1)[0]

      return self

  def get_best_chromosome(self):
      return self.best_chromosome_

  def get_best_features(self):
      return self.best_features_

In [ ]:
#read in dataset
import kagglehub
qsar_biodegradation_path = kagglehub.dataset_download("muhammetvarl/qsarbiodegradation")
print("Dataset successfully loaded.")

100%|██████████| 48.4k/48.4k [00:00<00:00, 35.6MB/s]

Extracting files...
Dataset successfully loaded.


In [ ]:
#create dataframe from path
biodeg = pd.read_csv(qsar_biodegradation_path+'/qsar-biodeg.csv')

X,y = biodeg.iloc[:,:-1].values, biodeg.iloc[:,-1].values

#normalize features
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
print(X.shape, y.shape)
genetic_alg = GeneticAlgorithm(
               model=KNeighborsClassifier(metric='minkowski'),
               population_size=30,
               num_generations=20,
               mutation_rate=0.05,
               crossover_rate=0.8,
               random_state=42)

genetic_alg.fit(X,y)
print("Best Features: ", genetic_alg.get_best_features())
print("Best Chromosome: ", genetic_alg.get_best_chromosome())

(1055, 41) (1055,)
Generation 1/20 | Best Score: 0.8145
Generation 2/20 | Best Score: 0.8408
Generation 3/20 | Best Score: 0.8408
Generation 4/20 | Best Score: 0.8408
Generation 5/20 | Best Score: 0.8408
Generation 6/20 | Best Score: 0.8445
Generation 7/20 | Best Score: 0.8445
Generation 8/20 | Best Score: 0.8445
Generation 9/20 | Best Score: 0.8445
Generation 10/20 | Best Score: 0.8445
Generation 11/20 | Best Score: 0.8445
Generation 12/20 | Best Score: 0.8502
Generation 13/20 | Best Score: 0.8507
Generation 14/20 | Best Score: 0.8559
Generation 15/20 | Best Score: 0.8559
Generation 16/20 | Best Score: 0.8559
Generation 17/20 | Best Score: 0.8607
Generation 18/20 | Best Score: 0.8607
Generation 19/20 | Best Score: 0.8682
Generation 20/20 | Best Score: 0.8682
Best Features:  [ 2  4  5  6 10 16 21 23 26 31 36 37]
Best Chromosome:  [0 0 1 0 1 1 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 1 0 0 0 0 1 0 0 0 0 1
 1 0 0 0]
